The purpose of this test is to simulate a real project of raw data processing. The goal is to create an analytical database that allows the government to send fines and deduct points to cars that have driven too fast on the highway. To do this, you have 3 files at your disposal:

passages.logs: a log file containing the following information:

the identifier of the passage in front of the radar,
the date of the passage,
the car registration number,
the recorded speed,
cars.json: a json file containing information about each car mentioned in the logs,
fines.csv:a table containing information about the possible fine levels.

To start, we will create a spark session:

(a) Run the following cell to connect to the session and retrieve the sparkContext.

In [ ]:
from pyspark.sql import SparkSession
from pyspark import SparkContext
from pyspark.sql.types import *

spark = SparkSession \
    .builder \
    .master("local[*]") \
    .appName("PySpark ETL") \
    .getOrCreate()

sc = spark.sparkContext
sc

Extract
In this part, the goal is to extract the information from our three data sources and return them as DataFrames.

(b) Define an extract function that takes no arguments (we'll assume the file names can't change) and returns a clean DataFrame (without duplicates or missing values) per source.

  It is recommended to first explore the datasets to determine the schema of each data source.

In [ ]:
# explore datasets
# help exploire naems of columns and stucture of data
for line in sc.textFile('passages.logs').take(5):
    print(line)
print('==============')
for line in sc.textFile('cars.json').take(15):
    print(line)
print('==============')
for line in sc.textFile('fines.csv').take(5):
    print(line)

In [ ]:
def extract():
    # passages logs file
    # split on / and read as an rdd string
    passages_RDD = sc.textFile('passages.logs')
    # deifne fucntion that split values and make sure numeric fields are integer
    def func_line(line):
        line = line.split(' / ')
        # field 0 is id integer, field 3 is the speed
        try:
            line[0] = int(line[0])
        except:
            line[0] = None
        try:
            line[3] = int(line[3])
        except:
            line[3] = None
        return line
    cleaned_passages_rdd = passages_RDD.map(func_line)
    # deifne shcme based on checking the datasets
    passages_schema = StructType([
        StructField('passage_id', IntegerType()),
        StructField('passage_date', StringType()),
        StructField('immatriculation', StringType()),
        StructField('speed', IntegerType())
    ])
    passages = spark.createDataFrame(cleaned_passages_rdd, passages_schema)
    passages = passages.dropDuplicates()
    # drop null values but only fomr columns where a null vlaue stop the row fomr being usefule, if passage_date is null it wont stop detecting a car
    passages = passages.na.drop(subset=['passage_id', 'immatriculation', 'speed'])

# cars jsin file
    # define schema for cars json
    cars_schema = StructType([
        StructField('immatriculation', StringType()),
        StructField('name', StringType()),
        StructField('location', StructType([
            StructField('address',StringType()),
            StructField('city',StringType()),
            StructField('state',StringType()),
            StructField('zipcode',StringType())
        ])),
        StructField('phone', StringType()),
        StructField('points', IntegerType())
    ])
    # read cars file wih the schema, multlinie chosed as file contain multiple jsonfiles
    cars = spark.read \
         .format('json') \
         .option('multiline', True) \
         .schema(cars_schema) \
         .load('cars.json')
    cars = cars.dropDuplicates()
    # drop a row if immatriculaton or points are empty, as it wil not hel indeniftying speedy cars
    cars = cars.na.drop(subset=['immatriculation', 'points'])

# fines.csv file
    # define fines shcema
    fines_schema = StructType([
        StructField('_c0', IntegerType()),
        StructField('low_terminal', IntegerType()),
        StructField('high_terminal', IntegerType()),
        StructField('file', IntegerType()),
        StructField('points', IntegerType()) ])
    fines = spark.read \
         .format('csv') \
         .option('header', True) \
         .schema(fines_schema) \
         .load('fines.csv')
    # _c0 is not needed
    fines = fines.drop('_c0')
    fines = fines.dropDuplicates()
    fines = fines.na.drop()

    return passages, cars, fines

Transform
In this part, the objective is to create a single DataFrame allowing to send the fines and to withdraw the necessary points to the drivers having driven too fast on the highway (i.e. a speed strictly higher than 130 km/h).

The columns we want to have are the following:

immatriculation: column identifying a vehicle
to_pay: sum of all the fines to be paid for this car
points_lost: sum of the points lost by this car
points : number of points that the person associated with this car had
withdrawal_license : boolean which is True if the person has no more points on his license

(c) Define a transform function performing the join between the different DataFrames and giving the final desired DataFrame.

  We could use join with an equality condition on the matriculation for passages and car and use a double inequality for the condition with fines.

In [ ]:
def transform (passages, cars, fines):
    # import functions needed for aggregation steps to create calcualted columns
    from pyspark.sql.functions import col, sum
    from pyspark.sql.window import Window
    #rename points column in fines table to avoid coillison with points columnj in cars tbale
    fines = fines.withColumnRenamed('points', 'fine_points')

    # keep cars speeding above 130 km/h
    passages2 = passages.filter(col('speed') > 130)
    # make excess column so that later fine levels match correctly
    passages2 = passages2.withColumn('excess' , col('speed') - 130)
    # left join passages woth cars on
    dataset = passages2.join(cars, on=['immatriculation'], how='left')
    # join with fines on double inequality and match excess between the low and high terminal
    dataset = dataset.join(fines, (dataset['excess'] > fines['low_terminal']) &
                          (dataset['excess'] <= fines['high_terminal']), how='left')
    # define window partition be immatculation to make one row per car
    w = Window().partitionBy('immatriculation')

    # adding colums using sum, window and over fucntions
    dataset = dataset.withColumn("to_pay", sum(col('file')).over(w))
    dataset = dataset.withColumn("points_lost", sum(col('fine_points')).over(w))

    # drop duplicate immatricualtion to make sure no cars are repeated
    dataset = dataset.dropDuplicates(['immatriculation'])
    # creat the boolean withfdrawal license column
    dataset = dataset.withColumn('withdrawal_license', (col('points') - col('points_lost')) <= 0)

    # select needed columns
    complete_dataset = dataset.select ('immatriculation', 'to_pay', 'points_lost', 'points', 'withdrawal_license')

    return complete_dataset

Verification

We are going to test that the above code works correctly.

(d) Run the following cell.

In [ ]:
passages, cars, fines = extract()
df = transform(passages, cars, fines)

display(df.limit(50).toPandas())